# Module T5-04: AlphaFold & Protein Design

**Tier 5 — Modern AI for Science | Module 04**

*Prerequisites: Tier 2 Module 06 (Protein Structure), Module T5-01 (LLM Fine-tuning — transformer architectures)*

GPU optional — structure prediction cells require a T4 GPU; all visualization and analysis cells run on CPU.

---

**By the end of this notebook you will be able to:**
1. Explain the AlphaFold2 architecture: MSA encoding, Evoformer, and structure module
2. Run AlphaFold2 via Google Colab and interpret pLDDT and PAE confidence metrics
3. Predict structures with ESMFold using only a language model (no MSA)
4. Evaluate predicted structures with TM-score and MolProbity
5. Understand the protein design loop: RFdiffusion → ProteinMPNN → AlphaFold validation

**Status:** 🚧 Placeholder — content coming soon.

**Key resources:**
- [AlphaFold Colab notebook](https://colab.research.google.com/github/deepmind/alphafold/blob/main/notebooks/AlphaFold.ipynb)
- [ColabFold (fast AF2 via MMseqs2)](https://github.com/sokrypton/ColabFold)
- [ESMFold — ESM Atlas](https://esmatlas.com/resources?action=fold)
- [RFdiffusion code and paper](https://github.com/RosettaCommons/RFdiffusion)
- [ProteinMPNN code](https://github.com/dauparas/ProteinMPNN)

---

[← Previous: Diffusion & Generative Models](../03_Diffusion_Generative_Models/03_Diffusion_Generative_Models.ipynb) | [Course Overview](../../README.md)

---

## 1. AlphaFold2 Architecture

> **TODO:** Describe the three-stage architecture:
> 1. **Input embeddings** — MSA (Multiple Sequence Alignment) representation + template features
> 2. **Evoformer** — 48 blocks of axial attention operating jointly on MSA rows/columns and pairwise representation
> 3. **Structure module** — invariant point attention (IPA) operating in 3D to predict backbone frames → atom coordinates
>
> Include a simplified diagram. Discuss how the pairwise representation encodes co-evolutionary information.

## 2. Running AlphaFold2 via ColabFold

> **TODO:** ColabFold uses MMseqs2 for fast MSA generation (replaces the slow HHblits/JackHMMer search). Walk through predicting a structure for a protein of interest. Show pLDDT and PAE output.

In [ ]:
# TODO: Install ColabFold
# !pip install colabfold[alphafold] -q

# Alternatively, run via the official Colab notebook:
# https://colab.research.google.com/github/sokrypton/ColabFold/blob/main/AlphaFold2.ipynb

# TODO: Submit a sequence for prediction
# query_sequence = 'MTEYKLVVVGAGGVGKSALTIQLIQNHFVDEYDPTIEDSY'  # KRAS fragment
# # Run via ColabFold API or local installation

## 3. Interpreting Confidence Metrics

> **TODO:** 
> - **pLDDT** (per-residue Local Distance Difference Test): 0–100, >90 = very high confidence, 50–70 = low (often disordered)
> - **PAE** (Predicted Aligned Error): error in Å for each residue pair when the other is used as reference. Low PAE between domains = confident relative position.
>
> Plot pLDDT along sequence. Visualize PAE heatmap. Color the structure in PyMOL/nglview by pLDDT.

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt

# TODO: Load AlphaFold2 output JSON and plot pLDDT
# with open('result_model_1.json') as f:
#     af2_data = json.load(f)
# plddt = af2_data['plddt']
# pae = np.array(af2_data['predicted_aligned_error'])

# fig, axes = plt.subplots(1, 2, figsize=(14, 5))
# axes[0].plot(plddt, color='steelblue')
# axes[0].axhline(90, color='green', linestyle='--', label='Very high (90)')
# axes[0].axhline(70, color='orange', linestyle='--', label='Confident (70)')
# axes[0].set_xlabel('Residue'); axes[0].set_ylabel('pLDDT'); axes[0].legend()
# axes[1].imshow(pae, cmap='Greens_r', vmin=0, vmax=30)
# axes[1].set_xlabel('Scored residue'); axes[1].set_ylabel('Aligned residue')
# axes[1].set_title('PAE (predicted aligned error)')
# plt.tight_layout(); plt.show()

## 4. ESMFold: Language-Model-Only Prediction

> **TODO:** ESMFold uses ESM-2 protein language model embeddings instead of an MSA. Faster than AF2, works for orphan sequences with no homologs. Query ESMFold API for the same sequence and compare pLDDT.

In [ ]:
import requests

# TODO: Query ESMFold API
# sequence = 'MTEYKLVVVGAGGVGKSALTIQLIQNHFVDEYDPTIEDSY'
# response = requests.post(
#     'https://api.esmatlas.com/foldSequence/v1/pdb/',
#     data=sequence,
#     headers={'Content-Type': 'application/x-www-form-urlencoded'}
# )
# pdb_str = response.text
# with open('esmfold_prediction.pdb', 'w') as f:
#     f.write(pdb_str)

## 5. Structure Quality Evaluation

> **TODO:** Compare AF2 and ESMFold structures by:
> - TM-score vs reference crystal structure (TM-align)
> - RMSD of backbone Cα atoms
> - MolProbity score (Ramachandran outliers, clashscore)

In [ ]:
from Bio.PDB import PDBParser, Superimposer
import numpy as np

# TODO: Calculate RMSD between two structures
# def calc_rmsd(pdb1, pdb2):
#     parser = PDBParser(QUIET=True)
#     s1 = parser.get_structure('pred', pdb1)
#     s2 = parser.get_structure('ref', pdb2)
#     sup = Superimposer()
#     atoms1 = [a for a in s1.get_atoms() if a.name == 'CA']
#     atoms2 = [a for a in s2.get_atoms() if a.name == 'CA']
#     n = min(len(atoms1), len(atoms2))
#     sup.set_atoms(atoms1[:n], atoms2[:n])
#     return sup.rms

# TODO: Run TM-align for TM-score
# !TMalign af2_prediction.pdb crystal_structure.pdb > tmalign_output.txt

## 6. Protein Design: RFdiffusion + ProteinMPNN

> **TODO:** Explain the two-step generative design pipeline:
> 1. **RFdiffusion** — diffusion model trained on protein backbone geometries. Generates novel backbone conformations for a target binding site (binder design) or from scratch (de novo design).
> 2. **ProteinMPNN** — inverse folding model. Given a fixed backbone, predicts the amino acid sequence most likely to fold into it.
> 3. **AlphaFold2 validation** — predict the structure of the ProteinMPNN sequence to verify it recapitulates the intended backbone.

In [ ]:
# TODO: RFdiffusion binder design example
# (requires GPU; run via the official Colab or local installation)
# !python RFdiffusion/scripts/run_inference.py \
#     inference.input_pdb=target.pdb \
#     'contigmap.contigs=[A1-100/0 50-100]' \
#     inference.num_designs=10 \
#     inference.output_prefix=designs/binder

# TODO: ProteinMPNN sequence design
# !python ProteinMPNN/protein_mpnn_run.py \
#     --pdb_path designs/binder_0.pdb \
#     --out_folder mpnn_sequences/ \
#     --num_seq_per_target 8 \
#     --sampling_temp 0.1

## 7. AlphaFold DB and Downstream Analysis

> **TODO:** Show how to query AlphaFold DB (https://alphafold.ebi.ac.uk/) via API. Download pre-computed structures for human proteome. Use predicted structures for binding site detection with fpocket or P2Rank.

In [ ]:
import requests

# TODO: Download structure from AlphaFold DB
# uniprot_id = 'P00533'  # EGFR
# url = f'https://alphafold.ebi.ac.uk/files/AF-{uniprot_id}-F1-model_v4.pdb'
# response = requests.get(url)
# with open(f'{uniprot_id}_alphafold.pdb', 'w') as f:
#     f.write(response.text)

## Summary

> **TODO:** Recap the AlphaFold2 revolution in structural biology. Discuss open questions: multimer complexes, dynamics, disordered regions, RNA. AlphaFold3 (2024) expansion to nucleic acids and small molecules. Protein design applications: enzyme engineering, binder design, vaccine development.

---

[← Previous: Diffusion & Generative Models](../03_Diffusion_Generative_Models/03_Diffusion_Generative_Models.ipynb) | [Course Overview](../../README.md)

---